# RAG without embeddings: MongoDB Atlas Search + Brave

The same question answering agent as [`mongodb-rag`](../mongodb-rag), over the
same AnyCompany 10-K, but with no embedding model. Instead of storing a Voyage
vector per chunk and querying it with `$vectorSearch`, the chunks are stored as
plain text and retrieved with **Atlas Search** (`$search`): Lucene full-text
search ranked by BM25. A second tool, **Brave Search**, covers what the filing
can't: accounting standards, regulations, or anything more recent.

What changes without embeddings:

- No embedding API key, cost, or latency, and nothing to re-embed when the
  chunking changes.
- Retrieval is lexical: it matches the query's words, stemmed (`leases` finds
  `lease`), not their meaning. Exact terms and figures match well; synonyms and
  other languages don't. The agent compensates by writing English keyword
  queries and retrying with the filing's vocabulary when a search misses.

Needs `MONGODB_CONNECTION` (an Atlas cluster; the free tier supports Atlas
Search) and `BRAVE_API_KEY` in `.env`.

In [12]:
import os
import re
import threading
import time
from pathlib import Path

import httpx
import pymupdf
import pymupdf4llm
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv()
YOUR_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-5-5"

PDF_PATH = Path("files/AnyCompany_financial_10K.pdf")

# A collection of its own: rag_db.test holds mongodb-rag's embedded chunks and
# its vector index, which this notebook leaves untouched.
DB_NAME = "rag_db"
COLLECTION_NAME = "text_search"
INDEX_NAME = "text_index"
INDEX_WAIT_TIMEOUT = 300

mongo_client = MongoClient(os.environ["MONGODB_CONNECTION"])
collection = mongo_client[DB_NAME][COLLECTION_NAME]
print(f"MongoDB server version: {mongo_client.server_info()['version']}")

brave_api_key = os.environ.get("BRAVE_API_KEY", "")
print(f"Brave API key available: {'Yes' if brave_api_key else 'No'}")

MongoDB server version: 8.0.34
Brave API key available: Yes


## 1. Load the 10-K as plain text

The PDF goes through the same steps as `mongodb-rag/main_new_chunking.py`
(drop duplicate pages, convert to markdown, split by header and then by size)
and stops before the embeddings. Two differences:

- Chunks are sized in characters: without an embedding model there is no
  token limit to respect.
- The section breadcrumb goes in its own `section` field instead of being
  prepended to the text, so the search can weight a match in the section title
  apart from a match in the body.

In [13]:
# Larger than mongodb-rag's 512 tokens: a lexical match needs the figures and
# the words that describe them in the same chunk.
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

HEADERS_TO_SPLIT_ON = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# The PDF's text already had markdown marks, so headers come out as
# '## ### Table: Market Risks' or '# **EXHIBITS...**'.
HEADER_PREFIX = re.compile(r"^[#\s]+")
HEADER_EMPHASIS = re.compile(r"[*~]+")

# PyMuPDF reads a false strikethrough in the PDF that splits words: 'Any ~~Com~~ pany'.
FALSE_STRIKETHROUGH = re.compile(r"\s?~~(.*?)~~\s?")
WHITESPACE = re.compile(r"\s+")


def unique_page_numbers(pdf_path):
    """Page numbers without repeats: this PDF carries every page twice.

    They are dropped before converting to markdown, not after: with both copies
    present, pymupdf4llm detects the headers differently in each and the same
    content ends up in different sections.
    """
    seen = set()
    numbers = []

    with pymupdf.open(pdf_path) as doc:
        for number, page in enumerate(doc):
            text = WHITESPACE.sub(" ", page.get_text()).strip()
            if text and text in seen:
                continue
            seen.add(text)
            numbers.append(number)

    return numbers


def clean_header(text):
    return HEADER_EMPHASIS.sub("", HEADER_PREFIX.sub("", text)).strip()


def chunk_pdf(pdf_path):
    pages = unique_page_numbers(pdf_path)
    markdown = FALSE_STRIKETHROUGH.sub(
        r"\1", pymupdf4llm.to_markdown(str(pdf_path), pages=pages)
    )
    sections = MarkdownHeaderTextSplitter(
        headers_to_split_on=HEADERS_TO_SPLIT_ON
    ).split_text(markdown)
    size_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )

    chunks = []
    seen = set()
    for section in sections:
        headers = [
            clean_header(section.metadata[key])
            for _, key in HEADERS_TO_SPLIT_ON
            if section.metadata.get(key)
        ]
        breadcrumb = " > ".join(header for header in headers if header)
        for piece in size_splitter.split_text(section.page_content):
            if not piece.strip() or (breadcrumb, piece) in seen:
                continue
            seen.add((breadcrumb, piece))
            chunks.append(
                {
                    "text": piece,
                    "section": breadcrumb,
                    "chunk_index": len(chunks),
                    "source": pdf_path.name,
                }
            )

    print(
        f"{len(pages)} unique pages, {len(markdown):,} characters, "
        f"{len(sections)} sections, {len(chunks)} chunks"
    )
    return chunks

## 2. Store the chunks and build the Atlas Search index

Two `string` fields analyzed with `lucene.english`, which lowercases, drops
stop words, and stems, so a query for "leases" also matches "lease". With
`dynamic: False` nothing else in the documents is indexed.

The collection is loaded once. Set `REBUILD = True` after changing the chunking
or the index definition: dropping the collection also drops its search index,
and both are rebuilt from scratch.

In [14]:
REBUILD = False

INDEX_DEFINITION = {
    "mappings": {
        "dynamic": False,
        "fields": {
            "text": {"type": "string", "analyzer": "lucene.english"},
            "section": {"type": "string", "analyzer": "lucene.english"},
        },
    },
}


def get_index():
    return next(iter(collection.list_search_indexes(INDEX_NAME)), None)


def wait_for_index():
    deadline = time.monotonic() + INDEX_WAIT_TIMEOUT
    while not (get_index() or {}).get("queryable"):
        if time.monotonic() > deadline:
            raise TimeoutError(
                f"Index '{INDEX_NAME}' still building after {INDEX_WAIT_TIMEOUT}s. "
                "Check its status in Atlas."
            )
        time.sleep(5)


if REBUILD:
    collection.drop()

if collection.estimated_document_count() == 0:
    chunks = chunk_pdf(PDF_PATH)
    collection.insert_many(chunks)
    print(f"Inserted {len(chunks)} chunks into {DB_NAME}.{COLLECTION_NAME}")
else:
    print(
        f"{DB_NAME}.{COLLECTION_NAME} already has "
        f"{collection.estimated_document_count()} chunks (REBUILD = True to reload)"
    )

if get_index() is None:
    collection.create_search_index(
        SearchIndexModel(definition=INDEX_DEFINITION, name=INDEX_NAME, type="search")
    )
    print(f"Index '{INDEX_NAME}' created, waiting for the initial build...")

wait_for_index()
print(f"Index '{INDEX_NAME}' is ready")

rag_db.text_search already has 111 chunks (REBUILD = True to reload)
Index 'text_index' is ready


## 3. The tools

`search_documents` runs a `compound` query: a document matches if it contains
any of the query's words, and ranks higher when they also appear close together
(`phrase`) or in its section title. BM25 already favors rare terms, so
"sublease" outweighs "total" without any tuning.

`web_search` calls the Brave Search API directly with `httpx`, with no MCP
server or Node.js involved.

In [15]:
@tool
def search_documents(query: str, limit: int = 8) -> str:
    """Full-text search over AnyCompany's 10-K filing.

    The search is lexical (BM25), not semantic: it matches the query's words,
    stemmed, against the filing. Write the query in English, as the keywords the
    filing would use ("operating lease liabilities sublease income"), not as a
    question. Search once per topic of a multi-part question.

    Args:
        query: English keywords to search for.
        limit: Maximum number of passages to return.
    """
    pipeline = [
        {
            "$search": {
                "index": INDEX_NAME,
                "compound": {
                    "should": [
                        {"text": {"query": query, "path": "text"}},
                        {
                            "phrase": {
                                "query": query,
                                "path": "text",
                                "slop": 3,
                                "score": {"boost": {"value": 3}},
                            }
                        },
                        {
                            "text": {
                                "query": query,
                                "path": "section",
                                "score": {"boost": {"value": 2}},
                            }
                        },
                    ],
                    "minimumShouldMatch": 1,
                },
            }
        },
        {"$limit": limit},
        {"$project": {"_id": 0, "text": 1, "section": 1}},
    ]

    results = list(collection.aggregate(pipeline))
    print(f"[search_documents] {query!r}: {len(results)} passages")

    if not results:
        return "No passages matched. Try other keywords or the filing's terminology."

    return "\n\n---\n\n".join(
        f"Section: {doc['section'] or '(none)'}\n{doc['text']}" for doc in results
    )


BRAVE_SEARCH_URL = "https://api.search.brave.com/res/v1/web/search"

# Brave's free plan allows one request per second, and Strands runs the tools of
# a turn concurrently: the lock spaces the requests out instead of getting 429s.
BRAVE_MIN_INTERVAL = 1.1
brave_lock = threading.Lock()
brave_last_request = 0.0

# Brave highlights the matched words in the snippets with <strong>.
HTML_TAG = re.compile(r"<[^>]+>")


@tool
def web_search(query: str, count: int = 5) -> str:
    """Search the web with Brave Search.

    Use it only for what AnyCompany's 10-K can't answer: accounting standards,
    regulations, definitions, industry context, or events after the filing.

    Args:
        query: The search query.
        count: Number of results to return (at most 20).
    """
    global brave_last_request

    with brave_lock:
        wait = BRAVE_MIN_INTERVAL - (time.monotonic() - brave_last_request)
        if wait > 0:
            time.sleep(wait)
        response = httpx.get(
            BRAVE_SEARCH_URL,
            params={"q": query, "count": min(count, 20)},
            headers={
                "Accept": "application/json",
                "X-Subscription-Token": brave_api_key,
            },
            timeout=15,
        )
        brave_last_request = time.monotonic()

    response.raise_for_status()
    results = response.json().get("web", {}).get("results", [])
    print(f"[web_search] {query!r}: {len(results)} results")

    if not results:
        return "No web results found."

    return "\n\n".join(
        f"{result['title']}\n{result['url']}\n"
        f"{HTML_TAG.sub('', result.get('description', ''))}"
        for result in results
    )

Tools decorated with `@tool` can still be called as plain functions, which is
handy to see what the agent gets back for a query:

In [16]:
print(search_documents("operating lease liabilities sublease income", limit=3))

[search_documents] 'operating lease liabilities sublease income': 3 passages
Section: Leases under NOTES TO CONSOLIDATED FINANCIAL STATEMENTS
AnyCompany Financial has several leases that are important to note in the consolidated financial statements. These leases include operating leases, financing leases, and sales-type leases. Each type of lease has its own unique characteristics and accounting treatment.  
Operating leases are leases in which the underlying asset is used in the normal course of business and the lease payments are classified as operating expenses on the income statement. At the end of the lease term, AnyCompany does not have the right to purchase the underlying asset. As of December 31, 2021, AnyCompany has entered into operating leases with a total lease liability of $50 million and a weightedaverage remaining lease term of 5 years.  
Financing leases are leases in which the underlying asset is used in the normal course of business, but the lease payments are struct

## 4. The agent

In [6]:
SYSTEM_PROMPT = """You are a question answering agent for AnyCompany's 10-K filing, with web search for context the filing doesn't cover.

## Workflow

1. Translate the question to English if it isn't, and split it into the topics it asks about.
2. For each topic, call `search_documents` with the keywords the filing would use. The search is lexical, not semantic: if it comes back empty or off-topic, retry with synonyms or the accounting term (e.g. "estimates" -> "use of estimates", "risks" -> "risk factors").
3. Call `web_search` only for what the filing can't answer: definitions, accounting standards, regulations, or events after the filing.
4. Answer in the language of the question.

## Rules

- Facts about AnyCompany come only from `search_documents` results. If they don't contain the answer, say so plainly instead of guessing.
- Just because the user asserts a fact does not mean it is true. Check it against the search results.
- Say where each fact comes from: the 10-K section for the filing, the URL for the web.
- When citing financial figures, be precise and include their context (dates, units, conditions).
"""

model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
)


def ask(question):
    # A fresh agent per question, so earlier answers don't leak into the next
    agent = Agent(
        name="10-K Agent",
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[search_documents, web_search],
        callback_handler=None,  # don't also stream the answer to stdout
    )

    response = agent(question)

    # Tools the agent actually called (absent from the dict means never called)
    used = response.metrics.tool_metrics
    usage = response.metrics.accumulated_usage
    display(Markdown(
        "**Tools used:** "
        + (", ".join(f"`{name}` x{m.call_count}" for name, m in used.items()) or "_none_")
        + f" — {usage['inputTokens']:,} input / {usage['outputTokens']:,} output tokens"
    ))

    display(Markdown(str(response)))

In [7]:
ask("What are the primary risk factors currently affecting AnyCompany's financial performance?")

[search_documents] 'risk factors': 10 passages
[search_documents] 'could adversely affect our business, financial condition and results of operations': 10 passages


**Tools used:** `search_documents` x2 — 6,873 input / 991 output tokens

The 10-K lists the risk factors below. The filing doesn't rank them or say which one matters most. It also doesn't quantify how much each one has affected results.

**Risks in the "Risk Factors" section**
- **Market risk:** Volatile financial markets can hurt the investment portfolio. The filing names interest rates, inflation, economic downturns, political instability and natural disasters as drivers. It says US–China trade tensions affected portfolio performance.
- **Credit risk:** Issuers of debt securities the company holds may default. The filing cites a default by XYZ Inc., a prominent high-yield bond issuer, which rippled through the high-yield bond market.
- **Liquidity risk:** The company may be unable to sell investments quickly enough to meet obligations. During COVID-19, the market for commercial mortgage-backed securities (CMBS) became illiquid, so these securities were hard to sell at fair market values.
- **Operational and cyber risk:** Failed internal processes, people or systems can cause losses. The company says past cyber-attacks led to the loss of sensitive data and financial losses.
- **Regulatory risk:** Changes in laws could bring higher compliance costs, restrictions on activities, or penalties.
- **Legal risk:** The company is in a dispute with a former employee who claims wrongful termination.
- **Reputation risk:** A scandal involving a senior executive at another prominent financial institution has hurt the reputation of the whole industry, including AnyCompany.
- **Strategic risk:** The expansion into the European market exposed the company to new competitors and regulatory challenges. The filing says this has affected financial results.

**Risks in the Notes to the Consolidated Financial Statements ("Certain Significant Risks and Uncertainties")**
- **Commodity prices:** Changes, such as in oil, could materially affect revenue and profitability.
- **Foreign exchange rates:** Changes could materially affect cost of goods sold and profitability.
- **Equity prices:** Changes could materially affect the investment portfolio and profitability.
- **Cybersecurity:** The company holds sensitive customer and financial data, so a major attack could materially affect its finances and reputation.
- **Regulatory compliance:** The company operates in a highly regulated industry. Even one compliance failure could bring fines, penalties and reputational damage.

The filing says management tries to reduce these risks through diversification, cybersecurity measures and compliance programs. It also says none of the risks can be eliminated entirely.


A question in Spanish: the English analyzer would never match "estimativos", so
the agent has to translate it before searching.

In [8]:
ask("Cuales son algunos de los estimativos para AnyCompany")

[search_documents] 'critical accounting estimates': 8 passages
[search_documents] 'use of estimates': 8 passages


**Tools used:** `search_documents` x2 — 7,381 input / 806 output tokens

Según la sección "Use of Estimates" de las Notas a los Estados Financieros Consolidados del 10-K, estos son los principales estimativos de AnyCompany:

| Categoría | Estimativo |
|---|---|
| Provisión para cuentas de dudoso recaudo | $100.000 (se estima que el 1% de las cuentas por cobrar será incobrable) |
| Valoración de inventarios (materia prima) | $50 por unidad de costo promedio para el próximo año |
| Depreciación: vida útil | 10 años (maquinaria de producción) |
| Depreciación: valor residual | 10% |
| Pasivo por garantías | $250.000 |
| Beneficios de pensión y posjubilación | Obligación de $5.000.000 |

**Cómo se calculó cada uno**
- **Cuentas de dudoso recaudo:** se basa en tendencias históricas, condiciones económicas actuales e información de clientes específicos.
- **Inventarios:** la compañía usa el método LIFO (últimas en entrar, primeras en salir). Como los precios de las materias primas son volátiles, estima el costo futuro de esos materiales a partir de tendencias históricas y condiciones de mercado.
- **Depreciación:** se aplica el método de línea recta. La vida útil y el valor residual se estiman evaluando la condición y la vida restante de los activos.
- **Garantías:** se considera el historial de garantías, el diseño del producto y los procesos de manufactura.
- **Pensiones y posjubilación:** se usan supuestos actuariales como tasas de descuento, aumentos salariales esperados y rotación de empleados.

**Inconsistencias del documento**
- Las notas dicen que los estados financieros se preparan bajo IFRS. Sin embargo, el método LIFO no está permitido por las IFRS, así que esas dos afirmaciones del filing no concuerdan.
- Las notas también dicen que la preparación de los estados financieros requiere estimaciones y supuestos basados en la mejor información disponible.


In [9]:
ask("What was the total operating lease liabilities and total sublease income of the AnyCompany as of December 31, 2021?")

[search_documents] 'sublease income': 8 passages
[search_documents] 'total operating lease liabilities December 31, 2021': 8 passages


**Tools used:** `search_documents` x2 — 11,836 input / 422 output tokens

**Total operating lease liabilities:** $50 million as of December 31, 2021. The weighted-average remaining lease term is 5 years. This comes from the "Leases" note in the Notes to Consolidated Financial Statements.

**Total sublease income:** I found no sublease income figure in the filing. My searches turned up nothing on sublease income for 2021.

The Leases note covers other lease types, but none of them is sublease income:
- Financing leases carry a liability of $30 million.
- Sales-type leases, where AnyCompany is the lessor, carry a lease receivable of $20 million.

The filing's related-party section mentions an office lease from PersonA, with annual rent of $500,000. That is rent AnyCompany pays, not sublease income.

The Leases note also says AnyCompany adopted ASC 842 on January 1, 2022. At that date it recorded a $100 million right-of-use asset and a $100 million lease liability.


The figures come from the filing and the standard comes from the web:

In [10]:
ask(
    "What were AnyCompany's operating lease liabilities as of December 31, 2021, "
    "and how does ASC 842 require a lessee to measure them?"
)

[search_documents] 'ASC 842 leases lessee present value of lease payments discount rate incremental borrowing rate': 8 passages
[search_documents] 'operating lease liabilities December 31, 2021': 8 passages
[web_search] 'ASC 842 lessee initial measurement lease liability present value of lease payments not yet paid discount rate implicit incremental borrowing rate': 4 results


**Tools used:** `search_documents` x2, `web_search` x1 — 21,627 input / 923 output tokens

**Operating lease liabilities at December 31, 2021**

The Leases note in the 10-K (Notes to Consolidated Financial Statements) reports operating lease liabilities of **$50 million** at December 31, 2021. The weighted-average remaining lease term was 5 years. For comparison, the same note reports financing lease liabilities of $30 million and sales-type lease receivables of $20 million. Those receivables are an asset, not a liability, because AnyCompany is the lessor on those leases.

The filing also says AnyCompany adopted ASC 842 on January 1, 2022. On that date it recorded a right-of-use (ROU) asset of $100 million and a lease liability of $100 million. The filing does not explain how the $50 million at year-end 2021 relates to the $100 million recorded at adoption.

**How ASC 842 requires a lessee to measure the liability**

The filing says only that lessees must recognize a lease liability and an ROU asset for all leases except short-term leases. It does not describe the measurement method. The following comes from outside sources (Deloitte DART and PwC Viewpoint guidance on ASC 842), not from the filing:
- At commencement, the lessee measures the lease liability as the **present value of the lease payments not yet paid**.
- The discount rate is the **rate implicit in the lease** if that rate is readily determinable.
- If the implicit rate is not readily determinable, the lessee uses its **incremental borrowing rate**.

Sources: AnyCompany 10-K, Leases note; https://dart.deloitte.com/USDART/home/codification/broad-transactions/asc842-10/roadmap-leasing/chapter-8-lessee-accounting/8-4-recognition-measurement; https://viewpoint.pwc.com/dt/us/en/pwc/accounting_guides/leases/leases__4_US/chapter_4_accounting_US/42_initial_recogniti_US.html


## Clean up

The collection and its search index stay in the cluster between runs. Atlas's
free tier allows only three search indexes per cluster, so drop them when
you're done:

In [11]:
# collection.drop()  # also drops the search index
mongo_client.close()